# **Welcome to our SentiShelter Kaggle Notebook!**

Please explore our analysis of the reddit dataset related to climate change and housing.

In [ ]:
#Validate our file paths (one dataset is the comments, the other is the posts)

# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# General Goals: 
* Clean data to just be about housing related ideas
* Remove any extraneous words in each comment
* Sentiment analysis for each subreddit
* Generate topic clusters
* Use spacy for easier analysis

# Step 0: Load and read the data

In [ ]:
#load in data (DO NOT run with bad wifi!)
climate_ds = pd.read_csv("/kaggle/input/the-reddit-climate-change-dataset/the-reddit-climate-change-dataset-comments.csv")

In [ ]:
#sanity check to make sure we are accessing data properly
climate_ds.head(3)

In [ ]:
climate_ds.info() #see stats about our dataset and what we're working with

# Reducing the dataset

In [ ]:
print(f"Total entries before filtering: {len(climate_ds)}")

# define housing related words
words_to_look_for = ['housing', 'landlord', 'house', 'home', 'apartment', 'rent', 'mortgage']

# create a regex, use word boundaries to prevent partial matching (like 'home' in 'homework')
pattern = '|'.join([f'\\b{word}\\b' for word in words_to_look_for])

# filter where the comment contains any of the words
climate_housing_ds = climate_ds[climate_ds['body'].str.contains(pattern, case=False, na=False)]

pd.set_option('display.max_colwidth', None) #wider columns to read comment
climate_housing_ds[['body']].head(3)

Other words that can be included
'real estate agent', 'broker', 'lease','down payment', 'property tax', 'appraisal', 
'foreclosure', 'short sale', 'escrow', 'closing costs', 'commission', 
'home inspection', 'pre-approval', 'debt-to-income ratio', 'adjustable-rate mortgage',
'inventory', 'zoning','capital gains tax', 'homeowners association', 'deed'

In [ ]:
#check our cleaned data
climate_housing_ds.info()
climate_housing_ds.head(3)

# Pick the most relevant subreddits
We wanted to narrow our dataset even further so we can actually run the analysis

In [ ]:
#get the subreddits that appear the most often
#pd.set_option('display.max_colwidth', None) #wider columns to read comment
#climate_housing_ds[['body']].head(3)

subreddit_counts = climate_housing_ds['subreddit.name'].value_counts()
print(subreddit_counts)

In [ ]:
#splice our dataset based on the top 5 (politics, worldnews, askreddit, collapse, canada)

subreddits_to_keep = ['canada']

# Filter the DataFrame to only include rows where 'subreddit.name' is in subreddits_to_keep
top_climate_housing_ds = climate_housing_ds[climate_housing_ds['subreddit.name'].isin(subreddits_to_keep)]

# Display the filtered DataFrame
top_climate_housing_ds.info()

Note: we ultamitely chose 'canada' as the final subreddit because it had a managable 3786 entries, so we could actually produce the analysis in a timely manner. Also, exploring the 'canada' subreddit would help us underestand housing sentiments in another country.

In [ ]:
top_climate_housing_ds.head(3) #check our newly spliced dataset

# Removing excess tokens

We need to make sure the comments are cleaned for later when we need to interpret topic clusters. This helps remove extraneous words like 'is' and 'or' which don't really contribute to the overall message of the comment

In [ ]:
#use stopwords to get rid of excess words
#this takes too long to run!
import re
import spacy
from spacy.pipeline import Lemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])

stopwords = nlp.Defaults.stop_words
len(stopwords)

nlp.Defaults.stop_words -= {"no", "not"}  # preserve negative sentiment

# Short function to process the text
def preprocess(comment):
    # Removing HTML tags and artifacts
    comment = re.sub(r'<.*?>', '', comment)  # remove HTML tags
    comment = re.sub(r'\n',' ', comment)  # remove newline characters
    comment = re.sub(r'/s',' ', comment)  # remove newline characters
    comment = re.sub(r'&[a-z]+;', ' ', comment)  # Remove HTML entities
    
    doc = nlp(comment)
    
    processed_words_list = [
        token.lemma_.lower() 
        for token in doc 
        if not token.is_punct and not token.is_stop and not token.like_url
    ]
    return ' '.join(processed_words_list)

# Apply the preprocessing function to your data
top_climate_housing_ds.loc[:, 'tokens'] = top_climate_housing_ds['body'].apply(preprocess)

In [ ]:
pd.set_option('display.max_colwidth', None) #wider columns to read comment
top_climate_housing_ds.head(3) #view our tokenized comments!

# Conver the timestamp
This will allow us to analyze the comments based on the date and time, and view overall trends over a time period

In [ ]:
#convert utc time
pd.reset_option('display.max_colwidth') #reset column size

#climate_housing_ds['date_time'] = pd.to_datetime(climate_housing_ds['created_utc'], unit='s')

top_climate_housing_ds.loc[:, 'date_time'] = pd.to_datetime(top_climate_housing_ds['created_utc'], unit='s')

top_climate_housing_ds.loc[:, 'year'] = top_climate_housing_ds['date_time'].dt.year
top_climate_housing_ds.loc[:, 'month'] = top_climate_housing_ds['date_time'].dt.month
top_climate_housing_ds.loc[:, 'day'] = top_climate_housing_ds['date_time'].dt.day

top_climate_housing_ds.loc[:, 'period'] = top_climate_housing_ds['year'].astype(str) + '-' + top_climate_housing_ds['month'].apply(lambda x: f'{x:02d}')

pd.set_option('display.max_colwidth', None)
top_climate_housing_ds.head(3)

# Part 1: Calculating the average sentiment
We grouped the comments by period and calculated the average sentiment for each

In [ ]:
pd.reset_option('display.max_colwidth') #reset view size

#calculate average sentiment per month
monthly_avg_sentiment = (top_climate_housing_ds.groupby(['period'])['sentiment'].mean()).reset_index()

#renaming the sentiment column to 'avg_sentiment'
monthly_avg_sentiment.rename(columns={'sentiment': 'avg_sentiment'}, inplace=True)

print(monthly_avg_sentiment)

# Visualizing the average sentiment

In [ ]:
#import packages
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import networkx as nx

#plot a scatter plot
sns.scatterplot(x='period', y='avg_sentiment', data=monthly_avg_sentiment, color='red', marker='o', s=20, label='Average Sentiment per Month')
plt.title('Scatterplot of Average Monthly Sentiment Analysis for Reddit Climiate Change Housing Comments')
plt.legend()
plt.show()

This scatterplot was not as visually helpful to understand what was happening with the sentiment over time!

In [ ]:
#plot a bar chart for all data
sns.barplot(x = 'period',
            y = 'avg_sentiment',
            data = monthly_avg_sentiment)
plt.title('Average Sentiment from January 2010 to August 2022')
# Show the plot
plt.show()

This is a much better plot. We also wanted to zoom in on a few specific years to see what has been done for ohter years.

In [ ]:
#plot a bar chart for 2010
year_2010_data = monthly_avg_sentiment[monthly_avg_sentiment['period'].str.contains('2010')]
sns.barplot(x = 'period',
            y = 'avg_sentiment',
            data = year_2010_data)
plt.title('Average Sentiment for January 2010')
# Show the plot
plt.show()

In [ ]:
#plot a bar chart for 2020
year_2020_data = monthly_avg_sentiment[monthly_avg_sentiment['period'].str.contains('2020')]
sns.barplot(x = 'period',
            y = 'avg_sentiment',
            data = year_2020_data)
plt.title('Average Sentiment for January 2020')
# Show the plot
plt.show()

In [ ]:
print(monthly_avg_sentiment.columns) #sanity check


We wanted to take a look at how each year compared to other years as well.

In [ ]:
#plot each year on top of one another as a line chart for comparison

# Ensure 'period' is in datetime format and set as index
monthly_avg_sentiment['period'] = pd.to_datetime(monthly_avg_sentiment['period'])
monthly_avg_sentiment.set_index('period', inplace=True)

# Pivoting the data
pivoted_data = monthly_avg_sentiment.pivot_table(index=monthly_avg_sentiment.index.month, 
                                             columns=monthly_avg_sentiment.index.year,
                                             values='avg_sentiment',
                                             aggfunc='mean')

# Plotting
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
for year in pivoted_data.columns:
    plt.plot(pivoted_data.index, pivoted_data[year], label=year)

plt.legend()
plt.title('Monthly Average Sentiment by Year')
plt.xlabel('Month')
plt.ylabel('Average Sentiment')
plt.show()

# Part 2: Most Frequent People and Places
This allows us to see what was most talked about in these comments, and relate them to tangigble everyday figures

In [ ]:
#extract human names using spacy!

import spacy
import pandas as pd

# Load the small English model
nlp = spacy.load('en_core_web_sm')

# Assuming your DataFrame is named top_climate_housing_ds
comments = top_climate_housing_ds[['tokens', 'sentiment']].dropna()  # Drop missing values

# Empty DataFrame to store names data
names_df = pd.DataFrame(columns=['Name', 'Count', 'Total_Sentiment'])

# Process each comment
for idx, row in comments.iterrows():
    doc = nlp(row['tokens'])
    for entity in doc.ents:
        if entity.label_ == 'PERSON':  # Check if the entity is a person's name
            name = entity.text
            
            # Check if the name is already in the DataFrame
            name_idx = names_df[names_df['Name'] == name].index
            
            if not name_idx.empty:
                name_idx = name_idx[0]
                names_df.at[name_idx, 'Count'] += 1
                names_df.at[name_idx, 'Total_Sentiment'] += row['sentiment']
            else:
                new_row = {'Name': name, 'Count': 1, 'Total_Sentiment': row['sentiment']}
                names_df = pd.concat([names_df, pd.DataFrame([new_row])], ignore_index=True)
            
            # Ensure 'Count' and 'Total_Sentiment' columns are numeric
            names_df['Count'] = names_df['Count'].astype(int)
            names_df['Total_Sentiment'] = names_df['Total_Sentiment'].astype(float)

# Calculate average sentiment
names_df['Avg_Sentiment'] = names_df['Total_Sentiment'] / names_df['Count']

# Sort by Count to get the top 50 names
top_50_names_df = names_df.nlargest(50, 'Count')

# Display the DataFrame
print(top_50_names_df)

Obviously, some of these are not names, but for the most part, the model captured some of the most frequently occurring names that would make sense, like Justin Trudeau (Prime Minister of Canada) and Andrew Scheer (Member of the House of Commons of Canada).

Next, we wanted to visualize this as a scatter plot and how they were associated with each sentiment

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Ensure you have the top 50 names DataFrame ready and sorted by 'Avg_Sentiment'
top_50_names_df.sort_values(by='Avg_Sentiment', ascending=False, inplace=True)

# Generate a color palette with the same number of colors as there are names
colors = sns.color_palette("coolwarm", n_colors=len(top_50_names_df))

plt.figure(figsize=(15, 10))

# Creating the scatter plot
sns.scatterplot(
    data=top_50_names_df,
    x='Count',
    y='Avg_Sentiment',
    hue='Name',  # Color by Name
    palette=colors,  # Apply the color palette
    s=100  # Set the size of the points
)

# Adding titles and labels
plt.title('Scatter Plot of Frequency vs. Average Sentiment Score for Top 50 Names')
plt.xlabel('Frequency (Count)')
plt.ylabel('Average Sentiment Score')

# Adjusting the legend
plt.legend(bbox_to_anchor=(1, 1), loc='upper left')

plt.xticks(rotation=45)
plt.tight_layout()

# Displaying the plot
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Ensure matplotlib is displaying inline
%matplotlib inline

# Ensure you have the top 50 names DataFrame ready
top_50_names_df.sort_values(by='Avg_Sentiment', ascending=False, inplace=True)

plt.figure(figsize=(15, 10))

# Creating the scatter plot
plot = sns.scatterplot(
    data=top_50_names_df,
    x='Count',
    y='Avg_Sentiment',
    s=100  # Set the size of the points
)

# Adding titles and labels
plt.title('Scatter Plot of Frequency vs. Average Sentiment Score for Top 50 Names')
plt.xlabel('Frequency (Count)')
plt.ylabel('Average Sentiment Score')

# Adding text labels
for i, row in top_50_names_df.iterrows():
    plot.text(row['Count'], row['Avg_Sentiment'], row['Name'], 
              horizontalalignment='right', 
              verticalalignment='top', 
              fontsize=10)

plt.xticks(rotation=45)
plt.tight_layout()

# Displaying the plot
plt.show()


In [ ]:
#repeat the same process but for locations

import spacy
import pandas as pd

# Load the small English model
nlp = spacy.load('en_core_web_sm')

# Assuming your DataFrame is named top_climate_housing_ds
comments = top_climate_housing_ds[['tokens', 'sentiment']].dropna()  # Drop missing values

# Empty DataFrame to store location data
locations_df = pd.DataFrame(columns=['Location', 'Count', 'Total_Sentiment'])

# Process each comment
for idx, row in comments.iterrows():
    doc = nlp(row['tokens'])
    for entity in doc.ents:
        if entity.label_ == 'GPE':  # Check if the entity is a location
            location = entity.text
            
            # Check if the location is already in the DataFrame
            location_idx = locations_df[locations_df['Location'] == location].index
            
            if not location_idx.empty:
                location_idx = location_idx[0]
                locations_df.at[location_idx, 'Count'] += 1
                locations_df.at[location_idx, 'Total_Sentiment'] += row['sentiment']
            else:
                new_row = {'Location': location, 'Count': 1, 'Total_Sentiment': row['sentiment']}
                locations_df = pd.concat([locations_df, pd.DataFrame([new_row])], ignore_index=True)
            
            # Ensure 'Count' and 'Total_Sentiment' columns are numeric
            locations_df['Count'] = locations_df['Count'].astype(int)
            locations_df['Total_Sentiment'] = locations_df['Total_Sentiment'].astype(float)

# Calculate average sentiment
locations_df['Avg_Sentiment'] = locations_df['Total_Sentiment'] / locations_df['Count']

# Sort by Count to get the top 50 locations
top_50_locations_df = locations_df.nlargest(50, 'Count')

# Display the DataFrame
print(top_50_locations_df)

In [ ]:
#make a scatterplot of the data

import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# sort the DataFrame by 'Avg_Sentiment'
top_50_locations_df.sort_values(by='Avg_Sentiment', ascending=False, inplace=True)

# generate a color palette with the same number of colors as there are locations
colors = sns.color_palette("coolwarm", n_colors=len(top_50_locations_df))

plt.figure(figsize=(15, 10))

sns.scatterplot(
    data=top_50_locations_df,
    x='Count',
    y='Avg_Sentiment',
    hue='Location',  # Color by Location
    palette=colors,  # Apply the color palette
    s=100  # Set the size of the points
)

# titles and labels
plt.title('Scatter Plot of Frequency vs. Average Sentiment Score for Top 50 Locations')
plt.xlabel('Frequency (Count)')
plt.ylabel('Average Sentiment Score')

plt.legend(bbox_to_anchor=(1, 1), loc='upper left')

plt.xticks(rotation=45)
plt.tight_layout()

plt.show()


# Part 3: Generating 20 topic clusters from comment text data
- Then reclassify post and comment dat
- Calcualte average sentiment
- Visualize clustesrs and keywords in each cluster
- Note: stop words have already been removed

In [ ]:
import gensim
from gensim import corpora
from gensim.models import LdaModel
from gensim.models import Phrases

# Preparing text data (ensure it is preprocessed)
texts = top_climate_housing_ds['tokens'].dropna().str.split()

# Creating bigrams
bigram = Phrases(texts, min_count=5)  # You can modify the parameters to suit your needs
texts = [bigram[line] for line in texts]

# Creating the term dictionary of our corpus, where every unique term is assigned an index
dictionary = corpora.Dictionary(texts)

# Converting a list of documents (corpus) into Document-Term Matrix using the dictionary prepared above.
doc_term_matrix = [dictionary.doc2bow(doc) for doc in texts]

# Creating the object for LDA model using gensim library
lda = LdaModel

# Running and Training LDA model on the document-term matrix
lda_model = lda(doc_term_matrix, num_topics=20, id2word=dictionary, passes=50)

# Results
topics = lda_model.print_topics(num_words=2)  # You can modify the number of words in each topic
for topic in topics:
    print(topic)

Note: we chose to use bigrams to reduce computational complexity, but it also may be worth tyring n grams to have a more accurate context for each topic. We also included the weights for the keywords so it is clear how much each word impacts the topic.

Now we need to reclassify our comments to fit one of the 20 topics

In [ ]:
#reclassify

def assign_topic_to_doc(lda_model, corpus):
    topics = []
    for doc in corpus:
        topic_probs = lda_model[doc]  # get topic probabilities for the document
        dominant_topic = max(topic_probs, key=lambda x: x[1])  # get the topic with the highest probability
        topics.append(dominant_topic[0])  # assign the topic number to the document
    return topics

# Applying the function to assign a dominant topic to each document
top_climate_housing_ds.loc[:, 'topic'] = assign_topic_to_doc(lda_model, doc_term_matrix)

# Displaying the DataFrame with the new 'dominant_topic' column
pd.set_option('display.max_colwidth', None)
top_climate_housing_ds.head()


Lastly, let's calculate the average sentiment of each topic cluster

In [ ]:
#average sentiment of each cluster

#calculate average sentiment per month
topic_avg_sentiment = (top_climate_housing_ds.groupby(['topic'])['sentiment'].mean()).reset_index()

#renaming the sentiment column to 'avg_sentiment'
topic_avg_sentiment.rename(columns={'sentiment': 'topic_sentiment'}, inplace=True)

#add in topics and associated keywords with weights
topics = lda_model.print_topics(num_words=2)
topics_keywords = [topic[1] for topic in topics]
topic_avg_sentiment['keywords'] = topics_keywords

print(topic_avg_sentiment)

In [1]:
def get_response(user_input):
    """
    Minimal wrapper function to connect the notebook to the chatbot.
    user_input: string typed by the user
    Returns: string response
    """
    # For example, return any comment that contains a keyword in the input
    keywords = user_input.lower().split()
    matches = climate_housing_ds[climate_housing_ds['body'].str.contains('|'.join(keywords), case=False, na=False)]
    
    if not matches.empty:
        # Return first matching comment as response
        return matches['body'].iloc[0]
    else:
        return "Sorry, I couldn't find a related comment in the dataset."
